## A/B Test 분석용 OLAP Cube 생성 + Tableau Public 시각화 실습

이번 실습에서는 이벤트 수준 Raw 데이터를 기반으로 A/B Test 분석용 Base Table과 Tableau 시각화용 OLAP Cube를 생성합니다.

#### 실습 목표

**Tableau용 데이터 준비**

- Raw 이벤트 테이블 생성

- A/B 분석용 Base Table 생성

- Tableau Public 시각화용 OLAP Cube 생성

- CSV Export 후 Tableau Public에서 활용

이번 실습은 실제 서비스 데이터를 단순화한 구조이기 때문에, 일부 지표 정의는 실무에서 사용되는 바와는 다를 수 있으며 데이터 구조 이해에 초점을 맞추고 진행하는 것이 중요합니다.

사용할 파일
- raw_sample_1000_users.csv
    - 이벤트 레벨 Raw 데이터

전체 처리 흐름
- Raw 데이터 로드
- 컬럼 정리
- Variant 생성
- Metadata 생성
- DuckDB 적재
- Base Table 생성
- OLAP Cube 생성
- CSV Export
- Tableau Public 시각화

## 1. 실습에 필요한 라이브러리 설치 및 import

In [1]:
import pandas as pd
import duckdb
import hashlib
import numpy as np
from pathlib import Path

## 2. Raw 데이터 로드

이번 실습에서는 raw_sample_1000_users.csv를 사용합니다.

이 데이터는 상품 단위 이벤트 로그 예시 데이터입니다.

In [2]:
raw_df = pd.read_csv("raw_sample_1000_users.csv")
raw_df.head()

C:\Users\SSAFY\AppData\Local\Temp\ipykernel_31032\4257100144.py:1: DtypeWarning: Columns (0: InvoiceNo) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_df = pd.read_csv("raw_sample_1000_users.csv")


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,revenue
0,536381,22139,RETROSPOT TEA SET CERAMIC 11 PC,23,2010-12-01 09:41:00,4.25,15311,United Kingdom,97.75
1,536381,84854,GIRLY PINK TOOL SET,5,2010-12-01 09:41:00,4.95,15311,United Kingdom,24.75
2,536381,22411,JUMBO SHOPPER VINTAGE RED PAISLEY,10,2010-12-01 09:41:00,1.95,15311,United Kingdom,19.50
3,536381,82567,"AIRLINE LOUNGE,METAL SIGN",2,2010-12-01 09:41:00,2.10,15311,United Kingdom,4.20
4,536381,21672,WHITE SPOT RED CERAMIC DRAWER KNOB,6,2010-12-01 09:41:00,1.25,15311,United Kingdom,7.50


In [3]:
print("row count:", len(raw_df))
print("columns:", raw_df.columns.tolist())

row count: 372340
columns: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country', 'revenue']


## 3. Raw 데이터 구조 확인

- 한 사용자가 여러 row를 가질 수 있음
- 한 row는 하나의 상품 구매 이벤트에 가까움
- 아직 A/B 실험군 정보는 없음
- 아직 사용자 메타데이터도 없음

즉, 바로 Tableau에서 대시보드를 만들기보다는 분석용 테이블 구조로 정리하는 과정이 필요합니다.

이 구조는 user-level이 아니라 event-level 데이터입니다.

즉,한 사용자가 여러 row를 가지며 각 row는 하나의 행동(이벤트)을 의미합니다.

따라서 이번 내용은
"사용자 기준 분석"이 아니라
"이벤트 기준 분석"을 중심으로 진행됩니다.


In [4]:
raw_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 372340 entries, 0 to 372339
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    372340 non-null  object 
 1   StockCode    372340 non-null  str    
 2   Description  372340 non-null  str    
 3   Quantity     372340 non-null  int64  
 4   InvoiceDate  372340 non-null  str    
 5   UnitPrice    372340 non-null  float64
 6   CustomerID   372340 non-null  int64  
 7   Country      372340 non-null  str    
 8   revenue      372340 non-null  float64
dtypes: float64(2), int64(2), object(1), str(4)
memory usage: 25.6+ MB


## 4. 컬럼 정리

이제 이후 분석에서 사용하기 쉽도록 컬럼 이름을 정리합니다.

변경할 컬럼
- CustomerID → user_id
- InvoiceDate → datestamp
- Quantity → purchased
- revenue → paidamount

실제 서비스에서는 click 이벤트가 별도로 존재하지만,이번 실습에서는 구조를 단순화하기 위해 모든 이벤트를 클릭으로 가정합니다.

또한 이번 실습에서는 각 row를 하나의 impression(노출)로 간주합니다.

여기서 purchased는 구매 여부가 아니라 구매 수량입니다.

하지만 통상적인 전환율(CVR) 분석을 위해, 수량이 1 이상이면 1(전환됨), 0이면 0으로 치환한 converted (전환 여부) 컬럼을 별도로 생성하여 사용합니다.

In [5]:
event_df = raw_df.rename(columns={
    "CustomerID": "user_id",
    "InvoiceDate": "datestamp",
    "Quantity": "purchased",
    "revenue": "paidamount"
}).copy()

event_df["datestamp"] = pd.to_datetime(event_df["datestamp"])


실제 데이터에는 클릭 정보가 없기 때문에, 구매 이벤트를 기준으로 클릭을 생성합니다.

즉, 구매가 발생한 이벤트는 반드시 클릭이 있었던 것으로 보고, 구매가 발생하지 않은 이벤트에 대해서는 일정 확률로 클릭을 생성하여 퍼널 구조를 구성합니다.

퍼널(impression → click → purchase)을 한 테이블에서 계산 가능하게 만들기 위한 보정하기 위해, InvoiceNo에 **544913_no_purchase**와 같은 형태의 데이터가 존재합니다.

In [6]:
import numpy as np
np.random.seed(42)

# 기본값 0
event_df["clicked"] = 0

# 구매 발생 row는 반드시 클릭
event_df.loc[event_df["purchased"] > 0, "clicked"] = 1

# 구매가 없는 row만 일부 클릭으로 설정
non_purchase_mask = event_df["purchased"] == 0
event_df.loc[non_purchase_mask, "clicked"] = np.random.binomial(
    1, 0.3, size=non_purchase_mask.sum()
)
# 구매 발생 여부(전환 여부)
event_df["converted"] = (event_df["purchased"] > 0).astype(int)

event_df.head()

,InvoiceNo,StockCode,Description,purchased,datestamp,UnitPrice,user_id,Country,paidamount,clicked,converted
0,536381,22139,RETROSPOT TEA SET CERAMIC 11 PC,23,2010-12-01 09:41:00,4.25,15311,United Kingdom,97.75,1,1
1,536381,84854,GIRLY PINK TOOL SET,5,2010-12-01 09:41:00,4.95,15311,United Kingdom,24.75,1,1
2,536381,22411,JUMBO SHOPPER VINTAGE RED PAISLEY,10,2010-12-01 09:41:00,1.95,15311,United Kingdom,19.50,1,1
3,536381,82567,"AIRLINE LOUNGE,METAL SIGN",2,2010-12-01 09:41:00,2.10,15311,United Kingdom,4.20,1,1
4,536381,21672,WHITE SPOT RED CERAMIC DRAWER KNOB,6,2010-12-01 09:41:00,1.25,15311,United Kingdom,7.50,1,1


## 5. 실제로 사용할 이벤트 컬럼만 선택

실습에서는 아래 컬럼만 사용합니다.

- user_id
- datestamp
- StockCode → item_id 역할
- clicked
- purchased
- paidamount

In [7]:
event_df = event_df[[
    "user_id",
    "datestamp",
    "StockCode",
    "clicked",
    "converted",
    "purchased",
    "paidamount"
]].rename(columns={
    "StockCode": "item_id"
})

## 6. Variant 생성

user_id 기준 hash 버킷팅으로 사용자를 control, test 그룹으로 나눕니다.

같은 사용자는 항상 같은 그룹에 들어가야 합니다.

In [8]:
def split_user(user_id):
    h = hashlib.md5(str(user_id).encode())
    return "test" if int(h.hexdigest(), 16) % 2 == 1 else "control"

In [9]:
user_variant_df = pd.DataFrame({
    "user_id": sorted(event_df["user_id"].unique())
})

user_variant_df["variant_id"] = user_variant_df["user_id"].apply(split_user)
user_variant_df.head()

,user_id,variant_id
0,12346,test
1,12348,test
2,12352,control
3,12359,control
4,12360,test


In [10]:
user_variant_df["variant_id"].value_counts()

variant_id
control    506
test       494
Name: count, dtype: int64

## 7. 사용자 메타데이터 생성

현재 Raw 데이터에는 성별, 연령 같은 사용자 메타데이터가 없기 때문에 교육용 실습을 위해 Python에서 추가 생성합니다.

여기서는 예시로 다음을 만듭니다.

gender: M / F
age: 20s / 30s / 40s / 50s

In [11]:
np.random.seed(42)

user_metadata_df = pd.DataFrame({
    "user_id": sorted(event_df["user_id"].unique()),
    "age": np.random.choice(["20s", "30s", "40s", "50s"], size=event_df["user_id"].nunique()),
    "gender": np.random.choice(["M", "F"], size=event_df["user_id"].nunique())
})

user_metadata_df.head()

,user_id,age,gender
0,12346,40s,F
1,12348,50s,M
2,12352,20s,M
3,12359,40s,M
4,12360,40s,M


## 8. 생성된 데이터 확인

이제 실습에 사용할 3개의 데이터프레임이 준비되었습니다.

event_df
user_variant_df
user_metadata_df

In [12]:
print("event_df rows:", len(event_df))
print("user_variant_df rows:", len(user_variant_df))
print("user_metadata_df rows:", len(user_metadata_df))

event_df rows: 372340
user_variant_df rows: 1000
user_metadata_df rows: 1000


## 9. DuckDB 연결 및 스키마 생성

이제 DuckDB를 사용해 Raw / Analytics 구조를 만듭니다.

- raw_data: 원천 데이터 저장
- analytics: Base Table, Cube 저장

In [13]:
con = duckdb.connect("ab_test_tableau.duckdb")

con.execute("CREATE SCHEMA IF NOT EXISTS raw_data;")
con.execute("CREATE SCHEMA IF NOT EXISTS analytics;")

print("DuckDB 연결 및 스키마 생성 완료")

DuckDB 연결 및 스키마 생성 완료


## 10. DataFrame을 DuckDB에 등록

Python에서 만든 DataFrame을 DuckDB에 등록한 뒤
SQL을 Python에서 실행하는 방식으로 테이블을 생성합니다.

In [14]:
con.register("tmp_event_df", event_df)
con.register("tmp_user_variant_df", user_variant_df)
con.register("tmp_user_metadata_df", user_metadata_df)

## 11. Raw 테이블 생성

이제 raw_data 스키마 아래에 3개의 Raw 테이블을 생성합니다.

- raw_data.user_event
- raw_data.user_variant
- raw_data.user_metadata

In [15]:
con.execute("DROP TABLE IF EXISTS raw_data.user_event;")
con.execute("DROP TABLE IF EXISTS raw_data.user_variant;")
con.execute("DROP TABLE IF EXISTS raw_data.user_metadata;")

In [16]:
con.execute("""
CREATE TABLE raw_data.user_event AS
SELECT * FROM tmp_event_df
""")

con.execute("""
CREATE TABLE raw_data.user_variant AS
SELECT * FROM tmp_user_variant_df
""")

con.execute("""
CREATE TABLE raw_data.user_metadata AS
SELECT * FROM tmp_user_metadata_df
""")

print("Raw 테이블 생성 완료")

Raw 테이블 생성 완료


## 12. Raw 테이블 적재 확인

In [17]:
con.execute("""
SELECT 'user_event' AS table_name, COUNT(*) AS row_count FROM raw_data.user_event
UNION ALL
SELECT 'user_variant' AS table_name, COUNT(*) AS row_count FROM raw_data.user_variant
UNION ALL
SELECT 'user_metadata' AS table_name, COUNT(*) AS row_count FROM raw_data.user_metadata
""").df()

,table_name,row_count
0,user_event,372340
1,user_variant,1000
2,user_metadata,1000


## 13. Raw 데이터 샘플 확인

In [18]:
con.execute("""
SELECT *
FROM raw_data.user_event
LIMIT 10
""").df()

,user_id,datestamp,item_id,clicked,converted,purchased,paidamount
0,15311,2010-12-01 09:41:00,22139,1,1,23,97.75
1,15311,2010-12-01 09:41:00,84854,1,1,5,24.75
2,15311,2010-12-01 09:41:00,22411,1,1,10,19.50
3,15311,2010-12-01 09:41:00,82567,1,1,2,4.20
4,15311,2010-12-01 09:41:00,21672,1,1,6,7.50
5,15311,2010-12-01 09:41:00,22774,1,1,24,30.00
6,15311,2010-12-01 09:41:00,22771,1,1,24,30.00
7,15311,2010-12-01 09:41:00,71270,1,1,1,1.25
8,15311,2010-12-01 09:41:00,22262,1,1,1,0.85
9,15311,2010-12-01 09:41:00,22637,1,1,1,2.55


In [19]:
con.execute("""
SELECT *
FROM raw_data.user_variant
LIMIT 10
""").df()

,user_id,variant_id
0,12346,test
1,12348,test
2,12352,control
3,12359,control
4,12360,test
5,12362,control
6,12372,test
7,12378,test
8,12379,test
9,12383,test


In [20]:
con.execute("""
SELECT *
FROM raw_data.user_metadata
LIMIT 10
""").df()

,user_id,age,gender
0,12346,40s,F
1,12348,50s,M
2,12352,20s,M
3,12359,40s,M
4,12360,40s,M
5,12362,50s,M
6,12372,20s,F
7,12378,20s,M
8,12379,40s,F
9,12383,30s,F


## 14. 분석용 Base Table 생성

이제 이벤트 로그, 실험군 정보, 사용자 메타데이터를 조인하여 분석용 Base Table을 생성합니다.

생성 테이블
- analytics.ab_base

In [21]:
con.execute("DROP TABLE IF EXISTS analytics.ab_base;")

In [22]:
con.execute("""
CREATE TABLE analytics.ab_base AS
SELECT
    e.user_id,
    CAST(e.datestamp AS DATE) AS event_date,
    e.datestamp,
    e.item_id,
    e.clicked,
    e.converted,
    e.purchased,
    e.paidamount,
    v.variant_id,
    m.age,
    m.gender
FROM raw_data.user_event e
JOIN raw_data.user_variant v
    ON e.user_id = v.user_id
LEFT JOIN raw_data.user_metadata m
    ON e.user_id = m.user_id
""")

print("analytics.ab_base 생성 완료")

analytics.ab_base 생성 완료


In [23]:
con.execute("""
SELECT *
FROM analytics.ab_base
LIMIT 10
""").df()

,user_id,event_date,datestamp,item_id,clicked,converted,purchased,paidamount,variant_id,age,gender
0,14502,2011-03-13,2011-03-13 12:39:00,22804,1,0,0,0.0,test,20s,F
1,17994,2011-06-03,2011-06-03 09:50:00,22693,1,0,0,0.0,control,20s,M
2,14057,2011-03-03,2011-03-03 14:56:00,23004,0,0,0,0.0,control,40s,M
3,14502,2011-04-18,2011-04-18 17:29:00,85049C,1,0,0,0.0,test,20s,F
4,18232,2011-07-05,2011-07-05 11:10:00,23085,0,0,0,0.0,control,50s,M
5,14400,2011-07-13,2011-07-13 15:50:00,21900,0,0,0,0.0,test,50s,F
6,17841,2011-06-21,2011-06-21 15:41:00,21790,1,0,0,0.0,test,30s,F
7,16066,2011-10-09,2011-10-09 11:53:00,23372,0,0,0,0.0,control,30s,M
8,17757,2011-07-10,2011-07-10 15:38:00,22439,0,0,0,0.0,control,30s,M
9,15910,2011-12-09,2011-12-09 10:51:00,23103,0,0,0,0.0,control,50s,M


## 15. Variant별 기본 KPI 확인

Cube를 만들기 전에 전체 기간 기준으로 control / test 그룹의 주요 KPI를 간단히 비교합니다.

확인 지표

- impressions
- users
- clicks
- purchases
- revenue
- CTR
- CVR
- ARPU

여기서 계산된 지표는 user-level 기준이 아니라 event-level 기준입니다.

예를 들어서
- user-level CVR: 전환한 사용자 비율
- event-level CVR: 클릭 이벤트 중 전환(conversion)이 발생한 비율

이 둘은 완전히 다른 의미를 가지므로 해석할 때 반드시 구분해야 합니다.

In [24]:
con.execute("""
SELECT
    variant_id,
    COUNT(*) AS impressions,
    COUNT(DISTINCT user_id) AS users,
    SUM(clicked) AS clicks,
    SUM(converted) AS conversions,
    SUM(purchased) AS purchases,
    SUM(paidamount) AS revenue,
    ROUND(SUM(clicked) * 1.0 / COUNT(*), 4) AS ctr,
    ROUND(SUM(converted) * 1.0 / NULLIF(SUM(clicked), 0), 4) AS cvr,
    ROUND(SUM(paidamount) * 1.0 / COUNT(DISTINCT user_id), 2) AS arpu
FROM analytics.ab_base
GROUP BY variant_id
ORDER BY variant_id
""").df()

,variant_id,impressions,users,clicks,conversions,purchases,revenue,ctr,cvr,arpu
0,control,181398,506,86226.0,45387.0,498930.0,831932.91,0.4753,0.5264,1644.14
1,test,190942,494,90765.0,47698.0,592639.0,981837.06,0.4754,0.5255,1987.52


## 16. 왜 OLAP Cube가 필요한가

원본 이벤트 로그를 Tableau에 그대로 넣을 수도 있지만, 보통은 그렇게 하지 않습니다.

이유는 다음과 같습니다.

- row 수가 많아질수록 시각화가 느려질 수 있다
- 필터를 바꿀 때마다 원본을 다시 집계해야 한다
- KPI 비교용 집계를 미리 만들어 두는 것이 효율적이다

Cube는 한 번 만들고 끝나는 게 아니라 새로운 데이터가 들어올 때마다 추가로 집계됩니다.

그래서 매번 전체 데이터를 다시 계산하는 것이 아니라, 새로 들어온 데이터만 집계해서 기존 결과에 붙이는 구조입니다.

그래서 이번 실습에서는 다음 2개의 집계 테이블을 만듭니다.

- analytics.ab_cube_daily
    - Tableau 시각화용
- analytics.ab_cube_daily_stats
    - 통계 계산 설명용

Cube는 단순히 속도를 위한 것이 아니라, "분석 기준(Dimension)"과 "지표(Measure)"를 분리해서 어떤 기준으로든 빠르게 집계할 수 있도록 만드는 구조입니다.

결국, Tableau에서 필터를 바꿔도 매번 raw 데이터를 다시 계산하지 않도록 하는 것이 핵심입니다.

## 17. Tableau 시각화용 OLAP Cube 생성

이번 테이블은 Tableau Public에서 바로 사용하기 쉬운 wide 형태의 일별 집계 테이블입니다.

집계 기준 Dimension

- event_date
- variant_id
- gender
- age

Measure

- impressions
- users
- clicks
- purchases
- revenue

In [25]:
con.execute("DROP TABLE IF EXISTS analytics.ab_cube_daily;")

In [26]:
con.execute("""
CREATE TABLE analytics.ab_cube_daily AS
SELECT
    event_date,
    variant_id,
    COALESCE(gender, 'unknown') AS gender,
    COALESCE(age, 'unknown') AS age,
    COUNT(*) AS impressions,
    COUNT(DISTINCT user_id) AS users,
    SUM(clicked) AS clicks,
    SUM(converted) AS conversions,
    SUM(purchased) AS purchases,
    SUM(paidamount) AS revenue
FROM analytics.ab_base
GROUP BY
    event_date,
    variant_id,
    COALESCE(gender, 'unknown'),
    COALESCE(age, 'unknown')
""")

print("analytics.ab_cube_daily 생성 완료")

analytics.ab_cube_daily 생성 완료


In [27]:
con.execute("""
SELECT *
FROM analytics.ab_cube_daily
ORDER BY event_date, variant_id, gender, age
LIMIT 20
""").df()

,event_date,variant_id,gender,age,impressions,users,clicks,conversions,purchases,revenue
0,2010-12-01,control,F,30s,86,1,36.0,20.0,111.0,360.05
1,2010-12-01,control,F,50s,185,3,92.0,48.0,369.0,970.25
2,2010-12-01,control,M,20s,442,2,213.0,116.0,418.0,964.39
3,2010-12-01,control,M,30s,85,1,43.0,23.0,416.0,950.09
4,2010-12-01,control,M,50s,239,4,124.0,61.0,786.0,1264.31
5,2010-12-01,test,F,20s,4,1,1.0,1.0,128.0,326.40
6,2010-12-01,test,F,30s,346,3,154.0,79.0,395.0,1136.78
7,2010-12-01,test,F,40s,90,2,48.0,25.0,219.0,582.60
8,2010-12-01,test,F,50s,370,3,175.0,86.0,371.0,770.83
9,2010-12-01,test,M,30s,4,1,2.0,1.0,8.0,79.60


이렇게 만들어진 Cube는 Tableau에서 바로 사용할 수 있는 형태입니다.

특히, 비율형 지표(CTR, CVR, ARPU)는 Cube에 넣지 않고 Tableau에서 Calculated Field로 계산하는 것이 일반적입니다.

→ 필터 변경 시 자동으로 재계산되기 때문입니다.

## 18. Tableau에서 사용할 KPI 계산 예시 확인

ab_cube_daily에는 기본 KPI만 넣고, 비율형 지표는 Tableau의 Calculated Field에서 계산하는 방식이 좋습니다.

예시

- CTR = clicks / impressions
- CVR = conversions / clicks
- ARPU = revenue / users

In [28]:
con.execute("""
SELECT
    event_date,
    variant_id,
    gender,
    age,
    impressions,
    users,
    clicks,
    conversions,
    purchases,
    revenue,
    ROUND(clicks * 1.0 / NULLIF(impressions, 0), 4) AS ctr,
    ROUND(conversions * 1.0 / NULLIF(clicks, 0), 4) AS cvr,
    ROUND(revenue * 1.0 / NULLIF(users, 0), 2) AS arpu
FROM analytics.ab_cube_daily
ORDER BY event_date, variant_id
LIMIT 20
""").df()

,event_date,variant_id,gender,age,impressions,users,clicks,conversions,purchases,revenue,ctr,cvr,arpu
0,2010-12-01,control,F,50s,185,3,92.0,48.0,369.0,970.25,0.4973,0.5217,323.42
1,2010-12-01,control,M,20s,442,2,213.0,116.0,418.0,964.39,0.4819,0.5446,482.20
2,2010-12-01,control,M,30s,85,1,43.0,23.0,416.0,950.09,0.5059,0.5349,950.09
3,2010-12-01,control,F,30s,86,1,36.0,20.0,111.0,360.05,0.4186,0.5556,360.05
4,2010-12-01,control,M,50s,239,4,124.0,61.0,786.0,1264.31,0.5188,0.4919,316.08
5,2010-12-01,test,M,30s,4,1,2.0,1.0,8.0,79.60,0.5000,0.5000,79.60
6,2010-12-01,test,M,50s,38,1,19.0,8.0,1676.0,3702.12,0.5000,0.4211,3702.12
7,2010-12-01,test,F,30s,346,3,154.0,79.0,395.0,1136.78,0.4451,0.5130,378.93
8,2010-12-01,test,F,20s,4,1,1.0,1.0,128.0,326.40,0.2500,1.0000,326.40
9,2010-12-01,test,M,40s,42,3,22.0,14.0,440.0,535.72,0.5238,0.6364,178.57


## 19. 통계 계산용 집계 테이블 생성

이번에는 n, sum, sum_of_squares를 저장하는 통계 계산용 Cube를 생성합니다.

생성 테이블
- analytics.ab_cube_daily_stats

이 테이블은 이번 실습에서 직접 Tableau 시각화에 사용하지는 않지만, 단순 시각화용이 아니라 통계 검정(Z-test 등)을 위한 데이터입니다.
n, sum, sum_of_squares를 이용하면 평균과 분산을 빠르게 계산할 수 있습니다.

In [29]:
con.execute("DROP TABLE IF EXISTS analytics.ab_cube_daily_stats;")

In [30]:
con.execute("""
CREATE TABLE analytics.ab_cube_daily_stats AS
SELECT
    event_date,
    variant_id,
    COALESCE(gender, 'unknown') AS gender,
    COALESCE(age, 'unknown') AS age,

    COUNT(*) AS row_count,
    COUNT(DISTINCT user_id) AS user_count,

    SUM(clicked) AS click_sum,
    SUM(converted) AS conversion_sum,
    SUM(purchased) AS purchase_sum,
    SUM(paidamount) AS revenue_sum,

    SUM(clicked * clicked) AS click_sq_sum,
    SUM(converted * converted) AS conversion_sq_sum,
    SUM(purchased * purchased) AS purchase_sq_sum,
    SUM(paidamount * paidamount) AS revenue_sq_sum
FROM analytics.ab_base
GROUP BY
    event_date,
    variant_id,
    COALESCE(gender, 'unknown'),
    COALESCE(age, 'unknown')
""")

print("analytics.ab_cube_daily_stats 생성 완료")

analytics.ab_cube_daily_stats 생성 완료


In [31]:
con.execute("""
SELECT *
FROM analytics.ab_cube_daily_stats
ORDER BY event_date, variant_id, gender, age
LIMIT 20
""").df()

,event_date,variant_id,gender,age,row_count,user_count,click_sum,conversion_sum,purchase_sum,revenue_sum,click_sq_sum,conversion_sq_sum,purchase_sq_sum,revenue_sq_sum
0,2010-12-01,control,F,30s,86,1,36.0,20.0,111.0,360.05,36.0,20.0,861.0,7.222163e+03
1,2010-12-01,control,F,50s,185,3,92.0,48.0,369.0,970.25,92.0,48.0,7149.0,5.777111e+04
2,2010-12-01,control,M,20s,442,2,213.0,116.0,418.0,964.39,213.0,116.0,4894.0,2.590668e+04
3,2010-12-01,control,M,30s,85,1,43.0,23.0,416.0,950.09,43.0,23.0,13776.0,8.117263e+04
4,2010-12-01,control,M,50s,239,4,124.0,61.0,786.0,1264.31,124.0,61.0,13428.0,3.430949e+04
5,2010-12-01,test,F,20s,4,1,1.0,1.0,128.0,326.40,1.0,1.0,16384.0,1.065370e+05
6,2010-12-01,test,F,30s,346,3,154.0,79.0,395.0,1136.78,154.0,79.0,6985.0,4.883254e+04
7,2010-12-01,test,F,40s,90,2,48.0,25.0,219.0,582.60,48.0,25.0,5299.0,5.047160e+04
8,2010-12-01,test,F,50s,370,3,175.0,86.0,371.0,770.83,175.0,86.0,5331.0,1.942211e+04
9,2010-12-01,test,M,30s,4,1,2.0,1.0,8.0,79.60,2.0,1.0,64.0,6.336160e+03


## 20. 통계 계산 예시

이제 집계된 n, sum, sum_of_squares를 사용하여 variant별 revenue 평균과 분산을 계산하는 예시를 확인합니다.

이렇게 계산된 평균과 분산은 이후 Z-test에서 사용되는 핵심 값입니다.

즉, Tableau에서는 시각화를 하고, 통계 검정은 별도로 수행하거나 이 데이터를 기반으로 계산하게 됩니다.

In [32]:
con.execute("""
WITH agg AS (
    SELECT
        variant_id,
        SUM(row_count) AS n,
        SUM(revenue_sum) AS revenue_sum,
        SUM(revenue_sq_sum) AS revenue_sq_sum
    FROM analytics.ab_cube_daily_stats
    GROUP BY variant_id
)
SELECT
    variant_id,
    n,
    revenue_sum,
    revenue_sq_sum,
    ROUND(revenue_sum * 1.0 / NULLIF(n, 0), 4) AS revenue_mean_per_row,
    ROUND((revenue_sq_sum * 1.0 / NULLIF(n, 0)) - POWER(revenue_sum * 1.0 / NULLIF(n, 0), 2), 4) AS revenue_variance_per_row
FROM agg
ORDER BY variant_id
""").df()

,variant_id,n,revenue_sum,revenue_sq_sum,revenue_mean_per_row,revenue_variance_per_row
0,control,181398.0,831932.91,7.592012e+07,4.5862,397.4944
1,test,190942.0,981837.06,6.282481e+09,5.1421,32876.1211


## 21. Cube 결과 검증

ab_cube_daily를 다시 variant 단위로 재집계하여 Base Table 결과와 유사한지 확인합니다.

In [33]:
con.execute("""
SELECT
    variant_id,
    SUM(impressions) AS impressions,
    SUM(users) AS users_non_distinct_sum,
    SUM(clicks) AS clicks,
    SUM(conversions) AS conversions,
    SUM(purchases) AS purchases,
    SUM(revenue) AS revenue
FROM analytics.ab_cube_daily
GROUP BY variant_id
ORDER BY variant_id
""").df()

,variant_id,impressions,users_non_distinct_sum,clicks,conversions,purchases,revenue
0,control,181398.0,1971.0,86226.0,45387.0,498930.0,831932.91
1,test,190942.0,1880.0,90765.0,47698.0,592639.0,981837.06


- impressions, clicks, purchases, revenue는 합산 가능
- users는 날짜/세그먼트 중복 가능성이 있으므로 해석 시 주의 필요

users는 DISTINCT 기준이 아니라 Cube 집계 과정에서 중복될 수 있기 때문에 단순 합산 시 실제 사용자 수와 다를 수 있습니다.

→ 따라서 user 수는 항상 DISTINCT 기준으로 해석해야 합니다.

## 22. CSV Export

이제 Tableau Public에 업로드할 수 있도록 집계 결과를 CSV 파일로 저장합니다.

저장 파일

- tableau_output/ab_cube_daily.csv
- tableau_output/ab_cube_daily_stats.csv

In [34]:
output_dir = Path("tableau_output")
output_dir.mkdir(exist_ok=True)

In [35]:
con.execute(f"""
COPY analytics.ab_cube_daily
TO '{output_dir / "ab_cube_daily.csv"}'
WITH (HEADER, DELIMITER ',')
""")

con.execute(f"""
COPY analytics.ab_cube_daily_stats
TO '{output_dir / "ab_cube_daily_stats.csv"}'
WITH (HEADER, DELIMITER ',')
""")

print("CSV export 완료")
print("저장 위치:", output_dir.resolve())

CSV export 완료
저장 위치: C:\Users\SSAFY\Desktop\TIL\09_data_analysis\c_abtest\chapter2\tableau_output


In [36]:
pd.read_csv(output_dir / "ab_cube_daily.csv").head()

,event_date,variant_id,gender,age,impressions,users,clicks,conversions,purchases,revenue
0,2011-10-09,control,M,30s,140,1,67,37,356,364.59
1,2011-12-09,control,M,50s,130,2,62,33,254,358.30
2,2010-12-17,control,M,20s,115,3,53,26,441,1134.89
3,2011-02-10,control,F,20s,159,3,81,43,4936,1490.70
4,2011-11-08,test,M,20s,85,2,38,22,243,372.13


In [37]:
pd.read_csv(output_dir / "ab_cube_daily_stats.csv").head()

,event_date,variant_id,gender,age,row_count,user_count,click_sum,conversion_sum,purchase_sum,revenue_sum,click_sq_sum,conversion_sq_sum,purchase_sq_sum,revenue_sq_sum
0,2011-03-03,control,M,40s,75,1,34,20,459,294.33,34,20,14697,5290.5465
1,2011-09-29,control,F,20s,143,1,66,35,565,615.39,66,35,12341,12865.0133
2,2011-04-10,control,M,20s,291,2,130,71,582,1074.94,130,71,29220,142288.4266
3,2011-06-24,control,M,40s,129,1,65,31,413,634.74,65,31,9365,17584.2476
4,2011-12-08,test,M,20s,267,2,137,67,235,518.91,137,67,1489,7529.0139


## 23. Tableau Public에서 활용할 파일

이제 Tableau Public에서는 다음 파일을 사용할 수 있습니다.

- tableau_output/ab_cube_daily.csv
  → Tableau 대시보드 시각화용

- tableau_output/ab_cube_daily_stats.csv
  → 통계 계산 구조 설명 및 확장용